In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os

In [2]:
# -----------------------------
# CONFIG (only thing you must keep consistent)
# -----------------------------

VARS_2D = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",
    "2m_temperature",
]

VARS_3D = [
    "geopotential",
    "specific_humidity",
    "u_component_of_wind",
    "v_component_of_wind",
    "vertical_velocity",
]

NLEVELS = 5
NTOD = 4


# -----------------------------
# channel metadata
# -----------------------------

def build_channel_names():
    names = []

    for v in VARS_2D:
        for t in range(NTOD):
            names.append(f"{v}_T{t}")

    for v in VARS_3D:
        for l in range(NLEVELS):
            for t in range(NTOD):
                names.append(f"{v}_L{l}_T{t}")

    return names


# -----------------------------
# load + average folds
# -----------------------------

def load_folds(base_dir, n_folds=5):
    folds = []
    for f in range(n_folds):
        path = os.path.join(base_dir, f"fold_{f}", "ig_results.pt")
        folds.append(torch.load(path, map_location="cpu"))
    return folds


def average_folds(folds, scalars, K=9):
    out = {}

    for s in scalars:
        chan = sum(f["channel"][s] for f in folds) / len(folds)
        spat = sum(f["spatial"][s] for f in folds) / len(folds)
        temp = sum(f["temporal"][s] for f in folds) / len(folds)

        out[s] = {"channel": chan, "spatial": spat, "temporal": temp}

    return out


# -----------------------------
# plotting
# -----------------------------

def plot_channel(vals, names, title):
    plt.figure(figsize=(14, 4))
    plt.bar(np.arange(len(vals)), vals.numpy())
    plt.xticks(np.arange(len(vals)), names, rotation=90, fontsize=6)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def plot_spatial(img, title):
    plt.figure()
    plt.imshow(img.numpy(), origin="lower", aspect="auto")
    plt.colorbar()
    plt.title(title)
    plt.show()


def plot_temporal(vals, title):
    plt.figure()
    plt.plot(vals.numpy(), marker="o")
    plt.xticks(range(NTOD), ["00","06","12","18"])
    plt.title(title)
    plt.show()


# -----------------------------
# main
# -----------------------------

def visualize(base_dir):
    scalars = ["mean", "sigma", "p_gt_2", "p_lt_-2"]

    folds = load_folds(base_dir)
    avg = average_folds(folds, scalars)

    channel_names = build_channel_names()

    target = 0  # choose among 9 biases

    for s in scalars:
        print(f"\n=== {s} ===")

        plot_channel(
            avg[s]["channel"][target],
            channel_names,
            f"{s} channel importance (target {target})"
        )

        plot_spatial(
            avg[s]["spatial"][target],
            f"{s} spatial importance (target {target})"
        )

        plot_temporal(
            avg[s]["temporal"][target],
            f"{s} temporal importance (target {target})"
        )